<a href="https://colab.research.google.com/github/cerr/pycerr-notebooks/blob/main/01_getting_started/pycerr_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Getting started with pyCERR: the PlanC data model

## Introduction

Everything in pyCERR lives in a **`PlanC`** container - lists of imaging objects: `scan` (CT/MR/PT/...), `structure` (segmentations), `dose`, `beams` (RT plans) and `deform` (registrations). This notebook loads a sample CT, tours those objects and their coordinate systems, creates structures and a dose from NumPy arrays, and visualizes a slice. No clinical RT data is needed - we synthesize a target, an OAR and a dose so the notebook is fully self-contained.

## Install pyCERR

In [ ]:
!pip install "pyCERR @ git+https://github.com/cerr/pyCERR.git"

## 1. Load a DICOM scan

`fetch_sample_data` downloads (and caches) a sample head & neck CT; `loadDcmDir` reads any folder of DICOM files into a `PlanC`.

In [ ]:
import cerr.plan_container as pc
from cerr import datasets

dcmDir = datasets.fetch_sample_data('head_and_neck')
planC = pc.loadDcmDir(dcmDir)
print('scan   :', len(planC.scan))
print('struct :', len(planC.structure))
print('dose   :', len(planC.dose))
print('beams  :', len(planC.beams))

## 2. The scan object

A `Scan` holds the 3-D pixel array plus the geometry needed to place it in physical space. `getScanXYZVals()` returns the x/y/z grid coordinates (cm) of the columns/rows/slices.

In [ ]:
scan = planC.scan[0]
arr = scan.getScanArray()              # (rows, cols, slices)
nR, nC, nS = (int(v) for v in scan.getScanSize())
xV, yV, zV = scan.getScanXYZVals()
print('array shape :', arr.shape, arr.dtype)
print('voxel (cm)  : %.2f x %.2f x %.2f'
      % (xV[1] - xV[0], yV[0] - yV[1], zV[1] - zV[0]))
print('Image2PhysicalTransM:\n', scan.Image2PhysicalTransM.round(3))

pyCERR carries three transforms per scan: `Image2PhysicalTransM` (voxel -> DICOM mm), `Image2VirtualPhysicalTransM` (voxel -> pyCERR virtual space) and `cerrToDcmTransM` (virtual -> DICOM).

## 3. Create structures from masks

`importStructureMask` turns a boolean/integer 3-D NumPy mask (on the scan grid) into a `Structure`. Here we build a spherical target (PTV) and a nearby organ-at-risk (Parotid).

In [ ]:
import numpy as np
Xc, Yc, Zc = np.meshgrid(xV, yV, zV, indexing='xy')
cx, cy, cz = xV[nC // 2], yV[nR // 2], zV[nS // 2]
distPTV2 = (Xc - cx) ** 2 + (Yc - cy) ** 2 + (Zc - cz) ** 2
ptvMask = distPTV2 < 3.0 ** 2
oarMask = ((Xc - (cx + 4.0)) ** 2 + (Yc - cy) ** 2
           + (Zc - cz) ** 2) < 1.5 ** 2
planC = pc.importStructureMask(ptvMask.astype(np.uint8), 0, 'PTV', planC)
planC = pc.importStructureMask(oarMask.astype(np.uint8), 0, 'Parotid', planC)
print([s.structureName for s in planC.structure])

## 4. Get a structure mask back

Contours are stored as polygons and rasterized on demand. `getStrMask` returns the binary 3-D mask on the scan grid.

In [ ]:
from cerr.contour import rasterseg as rs
mask = rs.getStrMask(0, planC)         # structure 0 = PTV
print('PTV mask:', mask.shape, '->', int(mask.sum()), 'voxels')

## 5. Add a dose distribution

`importDoseArray` attaches a dose grid (here synthetic: 70 Gy across the PTV with a Gaussian fall-off) associated with scan 0.

In [ ]:
PRESCRIPTION = 70.0
dist = np.sqrt(distPTV2)
dose3M = PRESCRIPTION * np.exp(-(np.clip(dist - 3.0, 0, None) ** 2)
                              / (2 * 2.5 ** 2))
dose3M[ptvMask] = PRESCRIPTION
planC = pc.importDoseArray(dose3M, xV, yV, zV, planC, 0,
                           {'fractionGroupID': 'Demo', 'units': 'GY'})
print('doses:', len(planC.dose))

## 6. Visualize a slice

In [ ]:
import matplotlib.pyplot as plt
from cerr.contour import rasterseg as rs

k = nS // 2
xV, yV, zV = planC.scan[0].getScanXYZVals()
ctSlc = planC.scan[0].getScanArray()[:, :, k]
doseSlc = planC.dose[0].doseArray[:, :, k]
ext = [xV[0], xV[-1], yV[-1], yV[0]]

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(ctSlc, cmap='gray', extent=ext)
dm = ax.imshow(np.ma.masked_where(doseSlc <= 1.0, doseSlc), cmap='jet',
               alpha=0.5, extent=ext)
plt.colorbar(dm, ax=ax, fraction=0.046, label='Dose (Gy)')
for strNum, color in [(0, 'r'), (1, 'c')]:
    m = rs.getStrMask(strNum, planC)[:, :, k].astype(float)
    ax.contour(xV, yV, m, levels=[0.5], colors=[color], linewidths=1.5)
ax.set_title('CT + structures + dose (axial slice %d)' % k)
plt.show()

## Next steps

* **Importing & exporting data** - `02_data_import/export_to_nifti_and_dicom.ipynb`
* **DVH & dose metrics** - `12_dose_analysis/dvh_and_dose_metrics.ipynb`
* Interactive 2D/3D viewing: `from cerr.viewer.pycerr_napari import showNapari` then `showNapari(planC)`, or the desktop GUI `from cerr.viewer.pycerr_gui import show` then `show(planC)`.